# SymPy 学习笔记（入门速通）

> 目标：掌握符号定义、表达式变形、微积分运算、方程求解与符号转数值。

SymPy（Symbolic Python）是纯 Python 编写的符号计算库，与 NumPy/SciPy 的数值计算形成互补。它的核心区别在于**精确性**：`math.sqrt(8)` 返回 `2.82842712475`，而 `sympy.sqrt(8)` 返回 `2*sqrt(2)`，保留数学结构而不做近似[citation:2][citation:4]。这使得 SymPy 适合推导公式、验证代数恒等式、生成 LaTeX 输出等场景。

**环境准备**
```bash
pip install sympy
```

In [2]:
import sympy
print("SymPy version:", sympy.__version__)

# 启用 Unicode 美化输出（Jupyter 中效果更好）
sympy.init_printing(use_unicode=True)

SymPy version: 1.14.0


## 1. 符号与表达式

SymPy 中所有变量必须先定义为 `Symbol` 对象，不能直接使用 Python 变量名。

| 函数 | 作用 | 说明 |
|---|---|---|
| `symbols('x y')` | 批量创建符号 | 返回多个 `Symbol`，空格分隔；支持 `real=True` 等假设 |
| `Symbol('x')` | 创建单个符号 | 适合动态生成变量名 |
| `Rational(1, 3)` | 精确有理数 | 返回 `1/3` 而非 `0.333...` |
| `sympify('x**2 + 1')` | 字符串转表达式 | 把字符串解析为 SymPy 表达式 |
| `.subs(old, new)` | 符号替换 | 替换变量或子表达式，返回新表达式 |
| `.evalf(n=15)` | 数值近似 | 将符号结果转为浮点数，默认 15 位精度 |

**核心特点**：表达式保持符号形式，自动做基础化简（如 `x - x = 0`），但不会自动展开或因式分解——需要显式调用对应函数[citation:2]。

**易错点**
- 未用 `symbols` 定义的 `x` 是普通 Python 变量，会直接报 `NameError`。
- `1/3` 在 Python 中先执行整数除法得 `0`，需用 `Rational(1, 3)` 或 `S(1)/3`。
- `.subs()` 返回**新表达式**，不会修改原对象。

In [3]:
from sympy import symbols, Rational, sympify, sqrt, pi, E

x, y = symbols('x y')
expr = x + 2*y
print("表达式:", expr)
print("自动化简 x - x:", expr - x)

# 精确数 vs 浮点数
print("sqrt(8) =", sqrt(8))          # 2*sqrt(2)，保持精确
print("sqrt(8).evalf() =", sqrt(8).evalf())
print("Rational(1,3) =", Rational(1, 3))

# 替换
print("subs x=1:", expr.subs(x, 1))   # 2*y + 1

# 字符串转表达式
e2 = sympify('x**2 + 3*x + 2')
print("sympify:", e2)

表达式: x + 2*y
自动化简 x - x: 2*y
sqrt(8) = 2*sqrt(2)
sqrt(8).evalf() = 2.82842712474619
Rational(1,3) = 1/3
subs x=1: 2*y + 1
sympify: x**2 + 3*x + 2


## 2. 表达式变形

符号计算最常用的操作：展开、因式分解、化简、合并同类项。

| 函数 | 作用 | 说明 |
|---|---|---|
| `expand(expr)` | 展开 | 分配律展开、三角函数展开（`trig=True`） |
| `factor(expr)` | 因式分解 | `expand` 的逆操作 |
| `simplify(expr)` | 通用化简 | 尝试多种策略，但不保证最优；可能较慢 |
| `trigsimp(expr)` | 三角化简 | 专门处理三角恒等式，如 `sin²+cos² → 1` |
| `collect(expr, x)` | 按变量合并 | 把表达式整理成 `x` 的多项式形式 |
| `apart(expr)` | 部分分式分解 | 有理式拆分为简单分式之和 |
| `together(expr)` | 通分 | `apart` 的逆操作 |

**`simplify` vs 专用函数**：`simplify` 是“万能但可能慢且结果不理想”的选择。如果明确知道表达式类型（三角、有理式），优先用 `trigsimp`、`factor`、`apart` 等专用函数[citation:8]。

**易错点**：`simplify` 的化简结果取决于内部启发式规则，不同版本可能有差异，不要依赖它做唯一正确输出。

In [4]:
from sympy import symbols, expand, factor, simplify, trigsimp, collect, apart, sin, cos

x, y = symbols('x y')

# 展开与因式分解
expr = (x + y)**2
print("expand:", expand(expr))
print("factor:", factor(x**2 + 2*x*y + y**2))

# 三角化简
trig_expr = sin(x)**2 + cos(x)**2
print("simplify:", simplify(trig_expr))
print("trigsimp:", trigsimp(trig_expr))

# 按变量合并
poly = x*y + x**2 + 3*x + 2
print("collect x:", collect(poly, x))

# 部分分式
frac = (2*x**2 + 3*x + 4) / (x + 1)
print("apart:", apart(frac))

expand: x**2 + 2*x*y + y**2
factor: (x + y)**2
simplify: 1
trigsimp: 1
collect x: x**2 + x*(y + 3) + 2
apart: 2*x + 1 + 3/(x + 1)


## 3. 微积分

SymPy 覆盖微积分核心操作，全部返回**精确符号结果**。

| 函数 | 作用 | 关键参数 |
|---|---|---|
| `limit(f, x, x0)` | 极限 | `dir='+'` 右极限，`dir='-'` 左极限；`x0` 可为 `oo`（无穷） |
| `diff(f, x)` | 导数 | 第三参数为阶数；多元函数可对多个变量依次求偏导 |
| `integrate(f, x)` | 不定积分 | 第二参数为积分变量 |
| `integrate(f, (x, a, b))` | 定积分 | 元组形式指定变量和上下限 |
| `series(f, x, x0, n)` | 级数展开 | 在 `x0` 处展开到 `n` 阶，带 `O()` 余项 |
| `summation(f, (n, a, b))` | 符号求和 | `b` 可为 `oo` |

**无穷符号**：SymPy 中用 `oo`（两个小写字母 o）表示正无穷，`-oo` 表示负无穷。

**易错点**
- `integrate` 对某些复杂函数可能返回未求值的形式（原样返回），需用 `.doit()` 或换用数值积分。
- `diff` 的阶数参数是整数，写在变量之后：`diff(f, x, 2)` 表示二阶导。

In [5]:
from sympy import symbols, limit, diff, integrate, series, summation, sin, exp, oo

x, n = symbols('x n')

# 极限
print("limit sin(x)/x:", limit(sin(x)/x, x, 0))          # 1
print("limit (1+1/x)^x:", limit((1 + 1/x)**x, x, oo))     # E

# 导数
f = x**3 + 2*x**2 + x
print("diff:", diff(f, x))          # 3*x**2 + 4*x + 1
print("二阶导:", diff(f, x, 2))     # 6*x + 4

# 积分
print("不定积分:", integrate(f, x))
print("定积分 ∫x^2 (0,1):", integrate(x**2, (x, 0, 1)))

# 级数展开
print("exp(x) 泰勒:", series(exp(x), x, 0, 5))

# 符号求和
print("Σ 1/n^2 (1,oo):", summation(1/n**2, (n, 1, oo)))

limit sin(x)/x: 1
limit (1+1/x)^x: E
diff: 3*x**2 + 4*x + 1
二阶导: 2*(3*x + 2)
不定积分: x**4/4 + 2*x**3/3 + x**2/2
定积分 ∫x^2 (0,1): 1/3
exp(x) 泰勒: 1 + x + x**2/2 + x**3/6 + x**4/24 + O(x**5)
Σ 1/n^2 (1,oo): pi**2/6


## 4. 方程求解

SymPy 提供多个求解器，选对函数很关键。

| 函数 | 适用场景 | 返回形式 |
|---|---|---|
| `solve(eq, x)` | 通用代数方程 | 解列表（可能为空） |
| `solveset(eq, x)` | 单方程（推荐） | `FiniteSet` / `Interval` / `ConditionSet` |
| `linsolve([eqs], [vars])` | 线性方程组 | 解元组集合 |
| `nonlinsolve([eqs], [vars])` | 非线性方程组 | 解元组集合 |
| `dsolve(eq, f(x))` | 微分方程 | 通解表达式 |

**`solve` vs `solveset`**：`solve` 是老接口，返回列表，可能遗漏解或返回不完整；`solveset` 是新接口，返回集合对象，语义更清晰，**官方推荐优先使用**[citation:15]。

**方程的表示**：`Eq(lhs, rhs)` 表示等式。但 SymPy 求解函数默认假设表达式等于 0，所以 `solve(x**2 - 1, x)` 等价于解 `x² - 1 = 0`。若方程为 `x² = 1`，可写 `Eq(x**2, 1)` 或直接 `x**2 - 1`[citation:15]。

**易错点**：`solve` 对多解方程可能返回部分解，`solveset` 更完整但返回集合，取值需用 `list(result)` 转换。

In [6]:
from sympy import symbols, solve, solveset, linsolve, nonlinsolve, Eq, FiniteSet

x, y, z = symbols('x y z')

# 单方程
print("solve x^2-5x+6:", solve(x**2 - 5*x + 6, x))
print("solveset:", solveset(x**2 - 5*x + 6, x))

# 含符号系数
a, b, c = symbols('a b c')
print("solve ax^2+bx+c:", solve(a*x**2 + b*x + c, x))

# 方程组
print("linsolve:", linsolve([x + y - 3, x - y - 1], [x, y]))
print("nonlinsolve:", nonlinsolve([x*y - 1, x - 2], [x, y]))

solve x^2-5x+6: [2, 3]
solveset: {2, 3}
solve ax^2+bx+c: [(-b - sqrt(-4*a*c + b**2))/(2*a), (-b + sqrt(-4*a*c + b**2))/(2*a)]
linsolve: {(2, 1)}
nonlinsolve: {(2, 1/2)}


## 5. 线性代数

`sympy.Matrix` 提供符号矩阵运算，支持精确分数和符号元素（不同于 NumPy 的浮点矩阵）。

| 方法/函数 | 作用 |
|---|---|
| `Matrix([[a, b], [c, d]])` | 创建矩阵 |
| `A.T` / `A.transpose()` | 转置 |
| `A.det()` | 行列式 |
| `A.inv()` | 逆矩阵（符号精确） |
| `A.eigenvals()` / `A.eigenvects()` | 特征值 / 特征向量 |
| `A.rref()` | 行简化阶梯形 |
| `A * B` | 矩阵乘法（`*` 即矩阵乘） |

**与 NumPy 的关键差异**：SymPy 的 `*` 是矩阵乘法，`A.multiply_elementwise(B)` 才是逐元素乘；NumPy 正好相反。

**性能提示**：符号矩阵运算随尺寸增长极快，超过 5×5 的符号矩阵通常计算会很慢，实际工程中数值矩阵优先用 NumPy。

In [ ]:
from sympy import Matrix, symbols, Rational

a, b, c, d = symbols('a b c d')
A = Matrix([[a, b], [c, d]])
print("A:\n", A)
print("det(A):", A.det())
print("A.T:\n", A.T)

# 数值矩阵的精确运算
B = Matrix([[1, 2], [3, 4]])
print("B.inv():\n", B.inv())   # 精确分数
print("B * B:\n", B * B)

# 特征值
print("eigenvals:", B.eigenvals())

## 6. 符号转数值

符号表达式最终常需要变成可计算的数值函数或数组，用于绘图或数值优化。

| 函数 | 作用 | 说明 |
|---|---|---|
| `.evalf(n)` | 求某个点的数值 | 对表达式整体求值，返回浮点数 |
| `.subs({...}).evalf()` | 多变量替换后求值 | 先代入具体值，再近似 |
| `lambdify(vars, expr, 'numpy')` | 转为 Python/NumPy 函数 | **最常用**，生成的函数可直接接受数组 |

**`lambdify` 的用法**：`f = lambdify(x, expr, 'numpy')` 后，`f(np.array([1,2,3]))` 返回 NumPy 数组，适合批量计算和绘图[citation:6]。

**易错点**：`lambdify` 的第二个参数是表达式，第一个参数是变量（或变量列表）。如果表达式含 `sympy` 特有函数（如 `zeta`），需确保 `numpy` 不支持时改用 `'math'` 或 `'scipy'` 模块。

In [ ]:
from sympy import symbols, lambdify, sin, exp
import numpy as np

x = symbols('x')
expr = sin(x) * exp(-x)

# evalf 单点求值
print("x=1 处:", expr.subs(x, 1).evalf())

# lambdify 转数值函数
f = lambdify(x, expr, 'numpy')
xs = np.linspace(0, 3, 5)
print("数组输入:", f(xs))

# 积分后转函数
from sympy import integrate
F = integrate(expr, x)
F_num = lambdify(x, F, 'numpy')
print("原函数在 x=1:", F_num(1))

## 7. 综合练习

1. 求函数 `f(x) = x³ - 3x + 1` 的驻点（`diff` + `solve`），并判断哪些是极小值点（二阶导 > 0）。
2. 计算 `∫₀^∞ e^{-x²} dx` 的精确值（提示：结果含 `sqrt(pi)`）。
3. 用 `linsolve` 解方程组 `2x + y = 4, 5x - 3y = 1`，并把结果转成浮点数。

**参考答案**

In [ ]:
from sympy import symbols, diff, solve, integrate, exp, oo, sqrt, pi, linsolve

x, y = symbols('x y')
f = x**3 - 3*x + 1

# 1. 驻点
df = diff(f, x)
critical = solve(df, x)
print("驻点:", critical)
for pt in critical:
    val = diff(f, x, 2).subs(x, pt)
    kind = "极小" if val > 0 else ("极大" if val < 0 else "拐点")
    print(f"  x={pt}, f''={val}, {kind}")

# 2. 高斯积分
gauss = integrate(exp(-x**2), (x, 0, oo))
print("∫e^(-x²) (0,∞) =", gauss)         # sqrt(pi)/2
print("数值:", gauss.evalf())

# 3. 线性方程组
sol = linsolve([2*x + y - 4, 5*x - 3*y - 1], [x, y])
print("linsolve:", sol)
print("浮点:", [s.evalf() for s in list(sol)[0]])

## 常见坑点总结

- **符号必须先定义**：`symbols('x')` 后才能使用 `x`，否则是普通 Python 变量。
- **`1/3` 不是精确分数**：用 `Rational(1,3)` 或 `S(1)/3`，否则先执行整数除法得 `0`。
- **`simplify` 不保证最优**：明确类型时用 `factor`、`trigsimp`、`apart` 等专用函数，结果更可控。
- **求解优先 `solveset`**：`solve` 是老接口，可能返回不完整解；`solveset` 语义更清晰[citation:15]。
- **SymPy 矩阵 `*` 是矩阵乘**：与 NumPy 相反，逐元素乘用 `multiply_elementwise`。
- **符号矩阵别太大**：超过 5×5 的符号矩阵运算会显著变慢，数值场景用 NumPy。
- **`lambdify` 是符号转数值的桥梁**：画图、优化、批量计算前，先把符号表达式转成 NumPy 函数。
- **`oo` 是两个字母 o**：表示无穷，`-oo` 负无穷，不要写成数字 `00`。